# BNL importer debug

In [92]:
# imports
from text_preparation.importers.lux.detect import detect_issues
from collections import namedtuple
import json, os
from datetime import date
import pandas as pd

## Detect.py

In [93]:
LuxIssueDir = namedtuple("IssueDirectory", ["provider", "alias", "date", "edition", "path"])
EDITIONS_MAPPINGS = {1: "a", 2: "b", 3: "c", 4: "d", 5: "e"}


In [94]:
# load json file /rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BNL/bnl_metadata.json
with open("/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BNL/bnl_metadata.json", "r") as f:
    bnl_metadata = json.load(f)
    

In [95]:
# take one example issue
bnl_metadata["actionfem"][0]['date']

'1936-05-15'

In [96]:
def entry_to_issue(alias: str, entry: dict) -> LuxIssueDir:
    """
    Convert one JSON entry into a LuxIssueDir.
    Example of entry:
        { "date": "1904-01-17", "local_path": "/mnt/.../1904-01-17_01" }
    """
    y, m, d = map(int, entry["date"].split("-"))

    # Detect edition from local_path (optional)
    # Example folder: 1904-01-17_01 → edition = 'a'
    edition = "a"
    base = os.path.basename(entry["local_path"])
    if "_" in base:
        suffix = base.split("_")[-1]
        if suffix.isdigit() and int(suffix) > 1:
            # map 2 → 'b', 3 → 'c', ...
            edition = EDITIONS_MAPPINGS.get(int(suffix), "a")

    return LuxIssueDir(
        provider="BNL",
        alias=alias,
        date=date(y, m, d),
        edition=edition,
        path=entry["local_path"],
    )

In [97]:
issue = entry_to_issue("actionfem", bnl_metadata["actionfem"][0])

In [98]:
issue

IssueDirectory(provider='BNL', alias='actionfem', date=datetime.date(1936, 5, 15), edition='a', path='/mnt/project_impresso/original/BNL/protected_034/1930756_newspaper_actionfem_1936-05-15_01')

In [99]:
def load_issues_from_json(json_path: str) -> list[LuxIssueDir]:
    """Load the full list of BNL issues from the precomputed JSON file.
    JSON structure:
    {
       "alias1": [ {"date": "...", "local_path": "..."} ],
       "alias2": [ ... ]
    }
    Args:
        json_path (str): Path to the JSON file.

    Returns:
        list[LuxIssueDir]: List of `LuxIssueDir` instances.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    issues = []
    for alias, entries in data.items():
        for entry in entries:
            issues.append(entry_to_issue(alias, entry))

    return issues

In [103]:
issues = load_issues_from_json("/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BNL/bnl_metadata.json")


## Check if compability with media list

In [107]:
# load bnl_metadata in a df with alias, date, local_path
bnl_metadata_df = pd.DataFrame(columns=["alias", "date", "local_path"])
rows = []
for alias, entries in bnl_metadata.items():
    for entry in entries:
        rows.append({"alias": alias, "date": entry["date"], "local_path": entry["local_path"]})

In [ ]:
df = pd.DataFrame(rows)
df

In [113]:
aliases = df['alias'].unique()
list(aliases), len(aliases)

(['actionfem',
  'arbeiter1878',
  'arlequin',
  'armeteufel',
  'avenirgdl',
  'beilagekirchlicheranzeiger',
  'buergerbeamten',
  'bulletinackerbauverein',
  'caecilia',
  'charribarri',
  'clarte',
  'courieresch',
  'courriergdl',
  'cycliste',
  'deletz1893',
  'deletzeburger1872',
  'demitock',
  'derarbeiter1889',
  'diamond',
  'diekwochen',
  'dossierinformatioundokumentatioun',
  'dunioun',
  'echo1890',
  'econom1871',
  'eschervolkszeitung',
  'escherzeitung',
  'floreal',
  'fortschrit1870',
  'freiejugend',
  'freihet',
  'gazgrdlux',
  'gemengemwinfo',
  'gewerkschaftler',
  'granducal',
  'gratislux',
  'gukuk',
  'handwerker',
  'hemecht',
  'hemecht1944',
  'hollywood',
  'indeplux',
  'jonghemecht',
  'jungekommunist',
  'jungewelt',
  'kampf',
  'keisecker',
  'keiseckerinfo',
  'kirchanzeiger1871',
  'kladderadatsch',
  'kommmit',
  'kommunistischegwerkschaftler',
  'kulturkampf',
  'land1866',
  'landwirth',
  'landwortbild',
  'laterne',
  'letzebuergerfleer',
  

In [117]:
df[df['alias'] == 'kommmit']['date'].min(), df[df['alias'] == 'kommmit']['date'].max()


('1884-01-15', '1884-12-01')

In [122]:
# for each alias, print min and max date
for alias in aliases:
    min_date = df[df['alias'] == alias]['date'].min()
    max_date = df[df['alias'] == alias]['date'].max()
    print(f"{alias}: {min_date} - {max_date}")

actionfem: 1927-10-15 - 1940-01-08
arbeiter1878: 1878-04-03 - 1881-12-24
arlequin: 1848-04-15 - 1848-05-10
armeteufel: 1903-11-29 - 1929-08-01
avenirgdl: 1868-04-21 - 1871-09-29
beilagekirchlicheranzeiger: 1946-01-15 - 1970-12-01
buergerbeamten: 1898-12-28 - 1916-12-28
bulletinackerbauverein: 1848-10-15 - 1876-12-15
caecilia: 1863-01-01 - 1871-12-01
charribarri: 1934-02-01 - 1938-02-25
clarte: 1945-05-26 - 1945-12-29
courieresch: 1895-03-30 - 1896-11-25
courriergdl: 1844-07-03 - 1868-12-27
cycliste: 1921-08-05 - 1921-12-30
deletz1893: 1893-01-01 - 1909-12-21
deletzeburger1872: 1872-05-05 - 1873-01-19
demitock: 1937-10-14 - 1940-05-10
derarbeiter1889: 1889-10-05 - 1890-10-08
diamond: 1919-03-18 - 1919-06-25
diekwochen: 1841-01-02 - 1848-12-30
dossierinformatioundokumentatioun: 1992-02-15 - 1997-08-15
dunioun: 1944-10-10 - 1948-04-03
echo1890: 1890-10-18 - 1897-12-26
econom1871: 1871-12-10 - 1872-04-21
eschervolkszeitung: 1884-05-17 - 1891-06-27
escherzeitung: 1889-01-06 - 1896-03-29
flo

In [146]:
duplicated_issues = df[df.duplicated(subset=['alias', 'date'])]

In [147]:
duplicated_issues['alias'].unique()

array(['armeteufel', 'avenirgdl', 'beilagekirchlicheranzeiger',
       'buergerbeamten', 'caecilia', 'courieresch', 'courriergdl',
       'deletz1893', 'diekwochen', 'echo1890', 'eschervolkszeitung',
       'fortschrit1870', 'gazgrdlux', 'gewerkschaftler', 'handwerker',
       'indeplux', 'jonghemecht', 'keisecker', 'keiseckerinfo',
       'kirchanzeiger1871', 'kladderadatsch',
       'kommunistischegwerkschaftler', 'kulturkampf', 'land1866',
       'landwirth', 'lunion', 'luxembourg1935', 'luxemburgerzeitung1868',
       'luxillustrierte', 'luxland', 'luxwort', 'luxzeit1844',
       'luxzeit1858', 'memoriala', 'memorialarlon', 'memorialc',
       'natioun', 'neuezeit1911', 'neuezeit1936', 'obermosel', 'omnibus',
       'ordo', 'proletarier', 'revue', 'socbotmem', 'socnatbul',
       'socnatfauna', 'sozrepublik', 'tageblatt', 'themecht',
       'verordnungsblatt', 'voixdesjeunes', 'volkfreu1869',
       'volksstimme1935', 'volkstribuene', 'waechtersauer', 'waeschfra',
       'wahrheit'

In [156]:
# check in df for alias 'themecht' and date '1984-10-01'
tmp = df[(df['alias'] == 'armeteufel') & (df['date'] == '1914-06-07')]
list(tmp['local_path'])

['/mnt/project_impresso/original/BNL/protected_027/1507564_newspaper_armeteufel_1914-06-07_02',
 '/mnt/project_impresso/original/BNL/protected_027/1507562_newspaper_armeteufel_1914-06-07_01']

In [155]:
duplicated_issues[duplicated_issues['alias'] == 'armeteufel']

,alias,date,local_path
439,armeteufel,1929-08-01,/mnt/project_impresso/original/BNL/protected_0...
805,armeteufel,1917-10-28,/mnt/project_impresso/original/BNL/protected_0...
820,armeteufel,1911-10-29,/mnt/project_impresso/original/BNL/protected_0...
843,armeteufel,1914-06-07,/mnt/project_impresso/original/BNL/protected_0...
1143,armeteufel,1914-04-05,/mnt/project_impresso/original/BNL/protected_0...
1152,armeteufel,1905-10-01,/mnt/project_impresso/original/BNL/protected_0...


## Classes.py

In [123]:
from text_preparation.importers.lux.classes import LuxNewspaperPage, LuxNewspaperIssue

In [159]:
issue_dir = entry_to_issue("cycliste", bnl_metadata["cycliste"][0])
big_issue = LuxNewspaperIssue(issue_dir)

In [160]:
big_issue.id, big_issue.edition, big_issue.alias, big_issue.path, big_issue.date, big_issue.issue_data, big_issue.pages, big_issue.image_properties, big_issue.ark_id

('cycliste-1921-11-26-a',
 'a',
 'cycliste',
 '/mnt/project_impresso/original/BNL-new/2025-impresso-3/cycliste/70795_1hq80kcks',
 datetime.date(1921, 11, 26),
 {'id': 'cycliste-1921-11-26-a',
  'cdt': '2025-12-04 17:47:18',
  'ts': '2025-12-04T16:47:18Z',
  'st': 'newspaper',
  'sm': 'print',
  'i': [{'m': {'id': 'cycliste-1921-11-26-a-i0003',
     'pp': [2],
     'tp': 'article',
     't': 'Sans titre',
     'lg': 'de',
     'ro': 1},
    'l': {'id': 'MODSMD_ARTICLE3',
     'parts': [{'comp_role': 'body',
       'comp_id': 'ART3-1',
       'comp_fileid': 'ALTO00002',
       'comp_page_no': 2},
      {'comp_role': 'body',
       'comp_id': 'ART3-2',
       'comp_fileid': 'ALTO00002',
       'comp_page_no': 2}]}},
   {'m': {'id': 'cycliste-1921-11-26-a-i0004',
     'pp': [2],
     'tp': 'article',
     't': 'Sans titre',
     'lg': 'de',
     'ro': 2},
    'l': {'id': 'MODSMD_ARTICLE4',
     'parts': [{'comp_role': 'body',
       'comp_id': 'ART4-1',
       'comp_fileid': 'ALTO00002',
 

In [136]:
new_issue_dir = entry_to_issue("jungekommunist", bnl_metadata["jungekommunist"][0])
new_issue = LuxNewspaperIssue(new_issue_dir)

In [137]:
new_issue.id, new_issue.edition, new_issue.alias, new_issue.path, new_issue.date, new_issue.issue_data, new_issue.pages, new_issue.image_properties, new_issue.ark_id

('jungekommunist-1921-04-15-a',
 'a',
 'jungekommunist',
 '/mnt/project_impresso/original/BNL-new/2025-impresso-3/jungekommunist/70795_z49r43rwzx',
 datetime.date(1921, 4, 15),
 {'id': 'jungekommunist-1921-04-15-a',
  'cdt': '2025-12-04 16:00:45',
  'ts': '2025-12-04T15:00:45Z',
  'st': 'newspaper',
  'sm': 'print',
  'i': [{'m': {'id': 'jungekommunist-1921-04-15-a-i0001',
     'pp': [1],
     'tp': 'article',
     't': 'Die deutsche Rlärzaktion und die kommunistische Jugend.',
     'lg': 'de',
     'ro': 1},
    'l': {'id': 'MODSMD_ARTICLE1',
     'parts': [{'comp_role': 'heading',
       'comp_id': 'ART2-1',
       'comp_fileid': 'ALTO00001',
       'comp_page_no': 1},
      {'comp_role': 'body',
       'comp_id': 'ART2-2',
       'comp_fileid': 'ALTO00001',
       'comp_page_no': 1},
      {'comp_role': 'body',
       'comp_id': 'ART2-3',
       'comp_fileid': 'ALTO00001',
       'comp_page_no': 1},
      {'comp_role': 'body',
       'comp_id': 'ART2-4',
       'comp_fileid': 'ALTO0